In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
import os

# Ensure we are always in the correct project directory
DRIVE_DIR = "/content/drive/MyDrive/FundGitHubProject"
if os.path.exists(DRIVE_DIR):
    os.chdir(DRIVE_DIR)
    print("✅ Successfully moved to project directory:", os.getcwd())
else:
    print("❌ Directory not found. Please ensure your Drive is mounted and the folder exists.")

✅ Successfully moved to project directory: /content/drive/MyDrive/FundGitHubProject


In [ ]:
# Pull the latest updates for your branch before starting work
!python gitfunctions/pull_branch.py matteo_branch

# 📘 Git Workflow Guide for ML Projects

Welcome to your personalized Git cheat sheet! Since you are working on evaluating models and fine-tuning them (e.g., using LoRA methods from the `eomt` experiments), you will be writing lots of code, testing hypotheses, and generating model weights.

This guide will show you how to safely manage your work on your dedicated branch (`matteo_branch`).

---

## 1️⃣ Saving Your Work (Commit & Push)
Whenever you reach a good stopping point, finish writing a new evaluation script, or tweak a LoRA parameter, you should save your work to GitHub.

### Method A: The Easy Way (Using your helper script)
Run this command in a new code cell. It will add all your changes, label them with your message, and push them to `matteo_branch`.
```python
!python gitfunctions/update_branch.py "Added LoRA fine-tuning loop for model evaluation"
```

### Method B: The Raw Git Way (Without the script)
If you want to know what the script is doing behind the scenes, these are the standard Git commands:
```bash
%%bash
# 1. 'Stage' all modified and new files to be saved
git add .

# 2. 'Commit' (save) the files with a descriptive message
git commit -m "Added LoRA fine-tuning loop for model evaluation"

# 3. 'Push' (upload) the saved changes to your branch on GitHub
git push origin matteo_branch
```

## 2️⃣ Getting the Latest Updates (Pull)
If you edited files on a different computer, or if a colleague updated a shared file, you need to pull those changes into your current Colab environment.

### Method A: The Easy Way
```python
# Pull updates specifically for your branch
!python gitfunctions/pull_branch.py matteo_branch
```

### Method B: The Raw Git Way
```bash
%%bash
# 1. Fetch the latest information from GitHub
git fetch origin

# 2. Make sure you are on your branch
git checkout matteo_branch

# 3. Download and merge the changes
git pull origin matteo_branch
```

## 3️⃣ Branching for Experiments (e.g., LoRA Fine-Tuning)
In Machine Learning, you often want to try a crazy new idea (like a new LoRA config) without breaking your working code. You do this by creating a **new branch** branching off from `matteo_branch`.

### Creating a new experiment branch
Let's say you want to try an experiment with adapter weights.
```bash
%%bash
# Create and switch to a new branch called 'matteo_lora_exp'
git checkout -b matteo_lora_exp
```
Now you can change files, train your model, and push normally. If the experiment fails, you can easily switch back to your safe branch and delete the experiment:
```bash
%%bash
# Switch back to your safe branch
git checkout matteo_branch

# (Optional) Delete the failed experiment branch locally
git branch -d matteo_lora_exp
```

## 4️⃣ Recovering History & Undoing Mistakes
Everyone makes mistakes! Here is how to fix common Git problems.

### Scenario A: "I changed some files but I hate the changes. I want to go back to how they were at my last commit!"
```bash
%%bash
# This throws away ALL uncommitted changes! Be careful!
git restore .
```

### Scenario B: "I want to look at the history of my commits"
```bash
%%bash
# Shows a neat list of your past commits and their ID numbers (hashes)
git log --oneline -n 5
```

### Scenario C: "I committed something, but I want to undo that commit (without losing the file changes)"
```bash
%%bash
# This undoes the last commit, but keeps your files exactly as they are right now
git reset --soft HEAD~1
```

## 5️⃣ Handling Large Machine Learning Files ⚠️
Since you are evaluating models and using LoRA, you will be generating **Large Files**:
*   `.safetensors`, `.bin`, `.pt`, `.pth` (Model Weights)
*   Large `.csv` or `.json` (Datasets)

**CRITICAL RULE:** NEVER run `git add .` if you have large model files in your folder, unless you are sure they are ignored by your `.gitignore` file. GitHub has a strict 100MB file limit. If you push a large model weight, it will freeze your repository.

**How to handle large files:**
1. Check your `.gitignore` file and ensure `*.safetensors` and `*.pth` are listed in it.
2. Save your trained models directly to Google Drive (e.g., in a dedicated `Saved_Models/` folder outside of the Git repository) instead of pushing them to GitHub.

In [7]:
# Run this cell to easily commit and push your current changes to 'matteo_branch'
# Feel free to change the commit message in the quotes below.
!python gitfunctions/update_branch.py "Update folder disposition"

--- Starting Update Process ---
> git add .
> git commit -m "Update folder disposition"
[main 3b0497d] Update folder disposition
 50 files changed, 47369 insertions(+), 59 deletions(-)
 delete mode 100644 coco-classes-mapping-master/coco_mapping_80to91.json
 delete mode 100644 coco-classes-mapping-master/coco_mapping_91to80.json
 create mode 100644 eomt/.gitignore
 create mode 100644 eomt/LICENSE
 create mode 100644 eomt/README.md
 rename =4.27.7 => eomt/__init__.py (100%)
 rename {configs => eomt/configs}/dinov2/cityscapes/semantic/eomt_base_640.yaml (100%)
 rename {configs => eomt/configs}/dinov2/coco/panoptic/eomt_base_640_2x.yaml (100%)
 create mode 100644 eomt/docs/index.html
 create mode 100644 eomt/docs/static/css/bulma-carousel.min.css
 create mode 100644 eomt/docs/static/css/bulma-slider.min.css
 create mode 100644 eomt/docs/static/css/bulma.css.map.txt
 create mode 100644 eomt/docs/static/css/bulma.min.css
 create mode 100644 eomt/docs/static/css/fontawesome.all.min.css
 crea

In [3]:
import os
from google.colab import userdata

# Set your Git identity
!git config --global user.email "s360426@studenti.unipi.it"
!git config --global user.name "MatteoAldovardi92"

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_TOKEN')

# Securely configure Git to use the token for all GitHub interactions
os.system(f'git config --global url."https://{github_token}@github.com/".insteadOf "https://github.com/"')

print("✅ Git identity and token configured successfully! You can now run the update cell.")

✅ Git identity and token configured successfully! You can now run the update cell.


In [5]:
%%writefile gitfunctions/update_branch.py
import os
import sys
import subprocess

def run_cmd(cmd):
    print(f"> {cmd}")
    os.system(cmd)

if __name__ == "__main__":
    commit_message = "Update from Colab"
    if len(sys.argv) > 1:
        commit_message = sys.argv[1]

    print("--- Starting Update Process ---")
    run_cmd("git add .")
    run_cmd(f'git commit -m "{commit_message}"')

    # Automatically get the current active branch
    current_branch = subprocess.getoutput("git rev-parse --abbrev-ref HEAD")

    print(f"Pushing to branch: {current_branch}")
    run_cmd(f"git push origin {current_branch}")
    print("--- Update Complete! ---")

Overwriting gitfunctions/update_branch.py


In [6]:
# 1. Unstage everything
!git reset

# 2. Append large file rules to .gitignore
with open('.gitignore', 'a') as f:
    f.write('\n# Ignore large ML files and datasets\n')
    f.write('*.safetensors\n*.bin\n*.pt\n*.pth\n*.h5\n*.csv\n*.json\nSaved_Models/\n')

# 3. Clear git cache so it respects the updated .gitignore for all files
!git rm -r --cached .

print('✅ Slate cleaned and .gitignore updated! Large files will now be ignored.\n➡️ You can now run the update cell above to push your changes.')

Unstaged changes after reset:
D	=4.27.7
D	configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
D	configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml
D	generate_mapping.py
M	gitfunctions/main.ipynb
M	gitfunctions/update_branch.py

It took 20.90 seconds to enumerate unstaged changes after reset.  You can
use '--quiet' to avoid this.  Set the config setting reset.quiet to true
to make this the default.
rm '.gitignore'
rm '=4.27.7'
rm 'LICENSE'
rm 'README.md'
rm 'Step4.ipynb'
rm 'Step4_Professional_Cleaned.ipynb'
rm 'coco-classes-mapping-master/README.md'
rm 'coco-classes-mapping-master/coco80.names'
rm 'coco-classes-mapping-master/coco91.names'
rm 'coco-classes-mapping-master/coco_mapping_80to91.json'
rm 'coco-classes-mapping-master/coco_mapping_91to80.json'
rm 'coco-classes-mapping-master/map_coco_classes.py'
rm 'configs/dinov2/cityscapes/semantic/eomt_base_640.yaml'
rm 'configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml'
rm 'docs/index.html'
rm 'docs/static/css/bulma-carousel.min.css'
r

In [10]:
import os

# 1. Create the new subfolder and navigate into it
os.makedirs('eomt/data', exist_ok=True)
# Save current directory to return later
base_dir = os.getcwd()
os.chdir('eomt/data')

# 2. Set credentials as environment variables for safe string handling
os.environ['CS_USER'] = "s360426@studenti.polito.it"
os.environ['CS_PASS'] = "UserProdRDD1!s"

print("Logging into Cityscapes...")
# 3. Login and save cookies
!wget -q --show-progress --keep-session-cookies --save-cookies=cookies.txt --post-data "username=$CS_USER&password=$CS_PASS&submit=Login" https://www.cityscapes-dataset.com/login/

print("\nDownloading Package 1 (gtFine_trainvaltest.zip)...")
# 4. Download first dataset
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=1

print("\nDownloading Package 3 (leftImg8bit_trainvaltest.zip - ~11GB)...")
# 5. Download second dataset
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3

# 6. Clean up the cookies file
if os.path.exists('cookies.txt'):
    os.remove('cookies.txt')

# Return to base directory
os.chdir(base_dir)
print("\n✅ Downloads completed in the eomt/data folder!")

Logging into Cityscapes...
index.html.2            [  <=>               ]  55.82K   260KB/s    in 0.2s    

gtFine_trainvaltest 100%[===================>] 240.87M  14.7MB/s    in 16s     

leftImg8bit_trainva 100%[===================>]  10.80G  22.1MB/s    in 9m 35s  

✅ Downloads completed in the eomt/data folder!


In [14]:
import os
import glob

data_dir = 'eomt/data'

print("🔍 Checking files and sizes in eomt/data:")
for file in sorted(os.listdir(data_dir)):
    filepath = os.path.join(data_dir, file)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f" - {file}: {size_mb:.2f} MB")

print("\n🧹 Running cleanup...")

# 1. Remove any file ending in .1 (or .html.2 etc)
# Using set() to avoid duplicates if a file matches both patterns
for f in set(glob.glob(f'{data_dir}/*.1') + glob.glob(f'{data_dir}/index.html*')):
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted duplicate/temp file: {f}")

# 2. Remove the original .zip files (the incomplete ones)
for f in glob.glob(f'{data_dir}/*.zip'):
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted incomplete zip: {f}")

# 3. Rename .zip.2 files to .zip
for f in glob.glob(f'{data_dir}/*.zip.2'):
    if os.path.exists(f):
        new_name = f.replace('.zip.2', '.zip')
        os.rename(f, new_name)
        print(f"Renamed: {f} -> {new_name}")

print("✅ Cleanup complete!")


🔍 Checking files and sizes in eomt/data:
 - gtFine_trainvaltest.zip: 240.87 MB
 - gtFine_trainvaltest.zip.2: 240.87 MB
 - index.html.2: 0.05 MB
 - leftImg8bit_trainvaltest.zip: 4676.86 MB
 - leftImg8bit_trainvaltest.zip.2: 11055.30 MB

🧹 Running cleanup...
Deleted duplicate/temp file: eomt/data/index.html.2
Deleted incomplete zip: eomt/data/gtFine_trainvaltest.zip
Deleted incomplete zip: eomt/data/leftImg8bit_trainvaltest.zip
Renamed: eomt/data/gtFine_trainvaltest.zip.2 -> eomt/data/gtFine_trainvaltest.zip
Renamed: eomt/data/leftImg8bit_trainvaltest.zip.2 -> eomt/data/leftImg8bit_trainvaltest.zip
✅ Cleanup complete!


In [15]:
import os

data_dir = 'eomt/data'

print("🔍 Final check of files and sizes in eomt/data:")
if os.path.exists(data_dir):
    for file in sorted(os.listdir(data_dir)):
        filepath = os.path.join(data_dir, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f" - {file}: {size_mb:.2f} MB")
else:
    print(f"Directory {data_dir} does not exist.")


🔍 Final check of files and sizes in eomt/data:
 - gtFine_trainvaltest.zip: 240.87 MB
 - leftImg8bit_trainvaltest.zip: 4676.86 MB


In [16]:
import os

data_dir = 'eomt/data'
bad_file = os.path.join(data_dir, 'leftImg8bit_trainvaltest.zip')

# 1. Delete the incomplete file
if os.path.exists(bad_file):
    os.remove(bad_file)
    print(f"🗑️ Deleted incomplete file: {bad_file}")

# Save current directory to return later
base_dir = os.getcwd()
os.chdir(data_dir)

# 2. Set credentials
os.environ['CS_USER'] = "s360426@studenti.polito.it"
os.environ['CS_PASS'] = "UserProdRDD1!s"

print("🔑 Logging into Cityscapes...")
# 3. Login and save cookies
!wget -q --show-progress --keep-session-cookies --save-cookies=cookies.txt --post-data "username=$CS_USER&password=$CS_PASS&submit=Login" https://www.cityscapes-dataset.com/login/

print("\n⬇️ Downloading Package 3 (leftImg8bit_trainvaltest.zip - ~11GB). This will take a while...")
# 4. Download Package 3
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3

# 5. Clean up the cookies file
if os.path.exists('cookies.txt'):
    os.remove('cookies.txt')

# Return to base directory
os.chdir(base_dir)
print("\n✅ Download of Package 3 completed in the eomt/data folder!")

🗑️ Deleted incomplete file: eomt/data/leftImg8bit_trainvaltest.zip
🔑 Logging into Cityscapes...
index.html              [  <=>               ]  55.82K   261KB/s    in 0.2s    

⬇️ Downloading Package 3 (leftImg8bit_trainvaltest.zip - ~11GB). This will take a while...
leftImg8bit_trainva 100%[===================>]  10.80G  26.8MB/s    in 7m 32s  

✅ Download of Package 3 completed in the eomt/data folder!


In [17]:
import os
import shutil

# Define paths
project_dir = '/content/drive/MyDrive/FundGitHubProject'
drive_data_dir = os.path.join(project_dir, 'eomt/data')
local_temp_dir = '/content/temp_download'

# Ensure directories exist
os.makedirs(drive_data_dir, exist_ok=True)
os.makedirs(local_temp_dir, exist_ok=True)

# Switch to local storage for the heavy download
os.chdir(local_temp_dir)

# Credentials
os.environ['CS_USER'] = "s360426@studenti.polito.it"
os.environ['CS_PASS'] = "UserProdRDD1!s"

print("🔑 Logging into Cityscapes (Local Environment)...")
!wget -q --show-progress --keep-session-cookies --save-cookies=cookies.txt --post-data "username=$CS_USER&password=$CS_PASS&submit=Login" https://www.cityscapes-dataset.com/login/

print("\n⬇️ Downloading Package 3 to LOCAL VM storage (This prevents Drive truncation)...")
!wget -q --show-progress --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3

local_file = 'leftImg8bit_trainvaltest.zip'
if os.path.exists(local_file):
    local_size = os.path.getsize(local_file) / (1024**3)
    print(f"\n✅ Local download complete. Size: {local_size:.2f} GB")

    print("\n🚚 Copying the 11GB file to Google Drive... (This might take a few minutes)")
    drive_path = os.path.join(drive_data_dir, local_file)

    # Using cp is often more reliable for large files in Colab than python's shutil
    !cp {local_file} "{drive_path}"

    if os.path.exists(drive_path):
        drive_size = os.path.getsize(drive_path) / (1024**3)
        print(f"✅ Copy complete. Drive file size: {drive_size:.2f} GB")

        if abs(local_size - drive_size) < 0.1:
            print("🎉 Success! The full 11GB file is now safely in your Drive.")
        else:
            print("⚠️ Warning: Sizes don't match exactly. Check the Drive folder.")
    else:
        print("❌ Failed to copy to Drive.")
else:
    print("❌ Download to local storage failed.")

# Cleanup local files to free up disk space on the VM
print("🧹 Cleaning up local temporary files...")
if os.path.exists(local_file):
    os.remove(local_file)
if os.path.exists('cookies.txt'):
    os.remove('cookies.txt')

# Return safely to project directory
os.chdir(project_dir)
print("Done!")

🔑 Logging into Cityscapes (Local Environment)...
index.html              [  <=>               ]  55.82K   250KB/s    in 0.2s    

⬇️ Downloading Package 3 to LOCAL VM storage (This prevents Drive truncation)...
leftImg8bit_trainva 100%[===================>]  10.80G  30.5MB/s    in 6m 17s  

✅ Local download complete. Size: 10.80 GB

🚚 Copying the 11GB file to Google Drive... (This might take a few minutes)
✅ Copy complete. Drive file size: 10.80 GB
🎉 Success! The full 11GB file is now safely in your Drive.
🧹 Cleaning up local temporary files...
Done!


In [18]:
import os

# Ensure we are in the project root
project_dir = '/content/drive/MyDrive/FundGitHubProject'
os.chdir(project_dir)

# Add .zip and the data folder to .gitignore
with open('.gitignore', 'a') as f:
    f.write('\n# Ignore compressed datasets and data folders\n')
    f.write('*.zip\neomt/data/\n')

# Clear git cache so it respects the new rules
print("🧹 Clearing git cache to apply new rules...")
!git rm -r --cached . > /dev/null

print("\n🔍 Checking git status (ensure .zip files are NOT listed under 'Untracked files'):")
!git status

print("\n✅ Slate cleaned! The 11GB file is now officially ignored. You can now run your git push cell.")

🧹 Clearing git cache to apply new rules...

🔍 Checking git status (ensure .zip files are NOT listed under 'Untracked files'):
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	deleted:    .gitignore
	deleted:    LICENSE
	deleted:    README.md
	deleted:    Step4.ipynb
	deleted:    Step4_Professional_Cleaned.ipynb
	deleted:    coco-classes-mapping-master/README.md
	deleted:    coco-classes-mapping-master/coco80.names
	deleted:    coco-classes-mapping-master/coco91.names
	deleted:    coco-classes-mapping-master/map_coco_classes.py
	deleted:    docs/index.html
	deleted:    docs/static/css/bulma-carousel.min.css
	deleted:    docs/static/css/bulma-slider.min.css
	deleted:    docs/static/css/bulma.css.map.txt
	deleted:    docs/static/css/bulma.min.css
	deleted:    docs/static/css/fontawesome.all.min.css
	deleted:    docs/static/css/index.css
	deleted:    docs/static/images/apple-touch-icon.png
	deleted: 

In [19]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	deleted:    .gitignore
	deleted:    LICENSE
	deleted:    README.md
	deleted:    Step4.ipynb
	deleted:    Step4_Professional_Cleaned.ipynb
	deleted:    coco-classes-mapping-master/README.md
	deleted:    coco-classes-mapping-master/coco80.names
	deleted:    coco-classes-mapping-master/coco91.names
	deleted:    coco-classes-mapping-master/map_coco_classes.py
	deleted:    docs/index.html
	deleted:    docs/static/css/bulma-carousel.min.css
	deleted:    docs/static/css/bulma-slider.min.css
	deleted:    docs/static/css/bulma.css.map.txt
	deleted:    docs/static/css/bulma.min.css
	deleted:    docs/static/css/fontawesome.all.min.css
	deleted:    docs/static/css/index.css
	deleted:    docs/static/images/apple-touch-icon.png
	deleted:    docs/static/images/arch.svg
	deleted:    docs/static/images/favicon-96x96.png
	deleted:    docs/static/images/favicon.ico
	